# Full Benchmark Run - 15 runtimes x 11 benchmarks (165 cells)

Kicks off the complete TQ-VLM-Bench matrix execution using the `Orchestrator`. Dual-GPU parallel execution and resume-from-checkpoint are enabled by default.

- 15 runtimes: baseline + 3 llama.cpp native KV quant + 8 TQ MSE + 3 TQ prod
- 11 benchmarks: 8 VLM + 3 Text
- Results stored at `../results/runs/*.json`
- Charts auto-generated at `../results/reports/`

In [ ]:
import os
import sys
from pathlib import Path

# Ensure llama.cpp shared libs are resolvable for the server subprocess
LCPP_LIB = Path('../../llama.cpp/build/bin').resolve()
os.environ['LD_LIBRARY_PATH'] = f"{LCPP_LIB}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Make tq_bench importable when running from notebooks/
sys.path.insert(0, str(Path('..').resolve()))

from tq_bench.config import load_runtimes, load_benchmarks, load_models
from tq_bench.orchestrator import Orchestrator
from tq_bench.reporters.export import export_csv, export_json
from tq_bench.reporters.summary import render_markdown_summary

In [ ]:
CONFIG_DIR = Path('../configs').resolve()
RESULTS_DIR = Path('../../results').resolve()

runtimes = load_runtimes(CONFIG_DIR / 'runtimes.yaml')
benchmarks = load_benchmarks(CONFIG_DIR / 'benchmarks.yaml')
models = load_models(CONFIG_DIR / 'models.yaml')

print(f'Runtimes:   {len(runtimes)}')
print(f'Benchmarks: {len(benchmarks)}')
print(f'Models:     {len(models)}')
print(f'Total cells per model: {len(runtimes) * len(benchmarks)}')

In [ ]:
MODEL_ID = 'Qwen3-VL-2B-Instruct'

orchestrator = Orchestrator(
    runtimes_path=CONFIG_DIR / 'runtimes.yaml',
    benchmarks_path=CONFIG_DIR / 'benchmarks.yaml',
    models_path=CONFIG_DIR / 'models.yaml',
    results_dir=RESULTS_DIR,
    model_id=MODEL_ID,
)

In [ ]:
# Dual-GPU parallel execution with resume support.
# Safe to re-run: completed cells are skipped via checkpoint.
orchestrator.run(parallel=True, resume=True)

In [ ]:
import json
import pandas as pd

runs_dir = RESULTS_DIR / 'runs'
rows = []
for p in sorted(runs_dir.glob('*.json')):
    with open(p) as f:
        r = json.load(f)
    rows.append({
        'runtime_id':   r.get('runtime_id'),
        'benchmark_id': r.get('benchmark_id'),
        'score':        r.get('score'),
        'n_samples':    r.get('n_samples'),
        'n_failed':     r.get('n_failed', 0),
        'wall_time_s':  r.get('wall_time_seconds'),
        'status':       r.get('status', 'ok'),
    })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} result cells')
df.head(20)

In [ ]:
# Export consolidated CSV/JSON
reports_dir = RESULTS_DIR / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

export_csv(df, reports_dir / 'full_run.csv')
export_json(df, reports_dir / 'full_run.json')
print(render_markdown_summary(df))

Charts (heatmap, degradation curves, VLM-vs-text scatter) are auto-generated at `../../results/reports/` by the orchestrator. See `03_analysis.ipynb` for interactive analysis and custom plots.